# Assignment 1 — Data Exploration and Cleaning: Truths

This notebook explores and cleans `truths.csv`, identifying data quality issues and applying validated corrections.

The final dataset will be used for integration with the other Truth Social datasets and subsequent assignments.

In [ ]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data"
)

DATA_PATH = DATA_DIR / "truths.csv"

# Load all columns as strings to preserve the original values
df_raw = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False
)

print("Rows:", len(df_raw))
print("Columns:", df_raw.shape[1])

display(df_raw.head())


## 1. Dataset Overview

We first examine the dataset structure, column names, data types, and missing values before applying any cleaning operations.

In [ ]:

print("Dataset shape:", df_raw.shape)

display(
    pd.DataFrame({
        "Column": df_raw.columns,
        "Data type": df_raw.dtypes.astype(str).values
    })
)

display(df_raw.head(5))


In [ ]:

# Count empty strings and common missing-value markers
missing_markers = ["", "NA", "N/A", "NULL", "NONE", "NAN"]

missing = pd.DataFrame({
    "Missing": [
        df_raw[col].str.strip().str.upper().isin(missing_markers).sum()
        for col in df_raw.columns
    ]
}, index=df_raw.columns)

missing["Percentage"] = (
    missing["Missing"] / len(df_raw) * 100
).round(2)

display(missing)


## 2. Duplicate Detection

We identify exact duplicate rows and duplicated post IDs.

Exact duplicates can be safely removed, while records sharing the same ID but containing different information require further investigation.

In [ ]:

# Count exact duplicates
exact_duplicates = df_raw.duplicated().sum()

# Remove exact copies to identify conflicting IDs
distinct_rows = df_raw.drop_duplicates()

id_counts = distinct_rows["id"].value_counts()
conflicting_ids = id_counts[id_counts > 1].index

print("Exact duplicates:", exact_duplicates)
print("IDs with conflicting records:", len(conflicting_ids))

# Inspect conflicting records
conflicting_records = distinct_rows[
    distinct_rows["id"].isin(conflicting_ids)
]

display(
    conflicting_records[
        ["id", "external_id", "url", "truth_retruthed"]
    ]
)


In [ ]:

# Create the working dataset
df_clean = df_raw.drop_duplicates().reset_index(drop=True).copy()

print("Original rows:", len(df_raw))
print("Removed duplicates:", len(df_raw) - len(df_clean))
print("Remaining rows:", len(df_clean))

assert df_clean.duplicated().sum() == 0


## 3. Data Type Conversion

The dataset was initially loaded as strings to preserve its original values.

We now convert identifiers, timestamps, boolean flags, and engagement metrics into appropriate data types.

In [ ]:

# Convert identifiers
for col in ["id", "author"]:
    df_clean[col] = pd.to_numeric(
        df_clean[col].replace("", pd.NA),
        errors="raise"
    ).astype("Int64")

# Convert timestamps
for col in ["timestamp", "time_scraped"]:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        format="ISO8601",
        errors="raise"
    )

# Convert boolean flags
for col in ["is_retruth", "is_reply"]:
    df_clean[col] = (
        df_clean[col]
        .map({"t": True, "f": False})
        .astype("boolean")
    )

# Convert engagement metrics
engagement_cols = ["like_count", "retruth_count", "reply_count"]

for col in engagement_cols:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="raise"
    ).astype("Int64")

display(df_clean.dtypes)


In [ ]:

# Check missing values after type conversion
display(df_clean.isna().sum())

# Check boolean values
for col in ["is_retruth", "is_reply"]:
    print(f"\n{col}")
    print(df_clean[col].value_counts(dropna=False))

# Verify dataset size
assert len(df_clean) == len(df_raw) - exact_duplicates


## 4. Missing Values and Author Recovery

After removing duplicates, 145 posts have a missing `author` value.

We attempt to recover these IDs by matching usernames with existing records in `truths.csv` and, subsequently, with `users.xml`.

Only unambiguous username-to-author mappings are used.

In [ ]:

# Extract usernames from post URLs
url_username = df_clean["url"].str.extract(
    r"(?i)truthsocial\.com/@([^/]+)/posts/",
    expand=False
).str.lower()

# Identify usernames stored in external_id
external_username = df_clean["external_id"].str.lower().where(
    df_clean["external_id"].str.fullmatch(
        r"(?i)[a-z0-9_]*[a-z_][a-z0-9_]*",
        na=False
    )
)

# Build username-author mappings from known records
known = pd.DataFrame({
    "username": url_username,
    "author": df_clean["author"]
}).dropna()

valid = known.groupby("username")["author"].nunique()
valid = valid[valid == 1].index

author_map = (
    known[known["username"].isin(valid)]
    .groupby("username")["author"]
    .first()
)

# Recover authors using available usernames
from_url = url_username.map(author_map)
from_external = external_username.map(author_map)

conflicts = (
    from_url.notna() & from_external.notna()
    & from_url.ne(from_external).fillna(False)
)

candidate = from_url.combine_first(from_external)
missing = df_clean["author"].isna()
recovered_internal = missing & candidate.notna() & ~conflicts

df_clean.loc[recovered_internal, "author"] = candidate[recovered_internal]

print("Recovered internally:", recovered_internal.sum())
print("Missing authors remaining:", df_clean["author"].isna().sum())


### 4.1 Cross-Dataset Recovery

The remaining missing authors are recovered using `users.xml`, which contains user IDs and usernames.

Matching is performed on normalized usernames, excluding ambiguous mappings.

In [ ]:

import xml.etree.ElementTree as ET

USERS_PATH = DATA_DIR / "users.xml"

root = ET.parse(USERS_PATH).getroot()

# Extract user IDs and usernames
users = pd.DataFrame([
    {
        "id_user": row.findtext("id_user"),
        "username": row.findtext("username")
    }
    for row in root.findall("row")
])

users["id_user"] = pd.to_numeric(
    users["id_user"], errors="coerce"
).astype("Int64")

users["username"] = users["username"].str.strip().str.lower()
users = users.dropna(subset=["id_user", "username"])
users = users[users["username"] != ""]

# Keep only unambiguous username mappings
valid = users.groupby("username")["id_user"].nunique()
valid = valid[valid == 1].index

users_map = (
    users[users["username"].isin(valid)]
    .groupby("username")["id_user"]
    .first()
)

# Recover remaining authors
username = url_username.combine_first(external_username)
candidate = username.map(users_map)

missing = df_clean["author"].isna()
recovered_external = missing & candidate.notna()

df_clean.loc[recovered_external, "author"] = candidate[recovered_external]

print("Recovered from users.xml:", recovered_external.sum())
print("Missing authors remaining:", df_clean["author"].isna().sum())


In [ ]:

print("AUTHOR RECOVERY SUMMARY")
print("-----------------------")
print("Initially missing:", 145)
print("Recovered internally:", recovered_internal.sum())
print("Recovered from users.xml:", recovered_external.sum())
print("Remaining missing:", df_clean["author"].isna().sum())

# Verify that recovered authors match the users dataset
recovered = recovered_internal | recovered_external

expected_author = username.map(users_map)

matches = (
    df_clean.loc[recovered, "author"]
    .eq(expected_author.loc[recovered])
)

print("Validated against users.xml:", matches.sum())

assert df_clean["author"].isna().sum() == 0
assert matches.fillna(False).all()


## 5. Text Quality and Cleaning

We identify missing, malformed, and potentially unusable post texts.

We also investigate recovery opportunities using other records and datasets, while preserving the original content.

In [ ]:

# Identify empty and malformed texts
text = df_clean["text"].str.strip()

empty_text = text.isin(["", "None", "NULL", "N/A", "NaN"])
malformed_text = text.eq("<")

problematic_text = empty_text | malformed_text

print("Empty or placeholder texts:", empty_text.sum())
print("Malformed texts:", malformed_text.sum())
print("Total problematic texts:", problematic_text.sum())

display(
    df_clean.loc[
        problematic_text,
        ["id", "text", "is_retruth", "is_reply"]
    ].head(10)
)


In [ ]:

# Build a mapping from IDs with valid texts
valid_text = df_clean.loc[
    ~problematic_text, ["id", "text"]
]

text_counts = valid_text.groupby("id")["text"].nunique()
unique_ids = text_counts[text_counts == 1].index

text_map = (
    valid_text[valid_text["id"].isin(unique_ids)]
    .groupby("id")["text"]
    .first()
)

# Find recoverable texts
text_candidate = df_clean["id"].map(text_map)
recoverable_text = problematic_text & text_candidate.notna()

print("Recoverable internally:", recoverable_text.sum())
print("Still unresolved:", (problematic_text & ~recoverable_text).sum())


In [ ]:

import arff
import re

ARFF_PATH = DATA_DIR.parent / "Additional_Data" / "additional_truths.arff"

with open(ARFF_PATH, encoding="utf-8-sig") as file:
    content = file.read()

# Handle unsupported escape sequences
content = re.sub(r"(?<!\\)\\x", r"\\\\x", content)

arff_data = arff.loads(content)
columns = [col[0] for col in arff_data["attributes"]]

additional_df = pd.DataFrame(arff_data["data"], columns=columns)

# Match external post identifiers
post_ids = df_clean["url"].str.extract(
    r"(?i)truthsocial\.com/@[^/]+/posts/(\d+)",
    expand=False
)

arff_ids = additional_df["external_id"].astype("string").str.strip()

matches = set(post_ids[problematic_text].dropna()) & set(arff_ids.dropna())

print("Matching post IDs in ARFF:", len(matches))


In [ ]:

MEDIA_PATH = DATA_DIR / "media.json"

media_df = pd.read_json(MEDIA_PATH)

media_ids = pd.to_numeric(
    media_df["truth_id"], errors="coerce"
).dropna()

df_clean["has_media"] = df_clean["id"].isin(media_ids)

print("Problematic texts:", problematic_text.sum())
print("With media:", (problematic_text & df_clean["has_media"]).sum())
print("Without media:", (problematic_text & ~df_clean["has_media"]).sum())


In [ ]:

import html

# Create a clean copy of the text column
df_clean["text_clean"] = df_clean["text"].copy()

# Decode HTML entities only where necessary
has_entities = df_clean["text_clean"].str.contains(
    r"&(?:#\d+|#x[0-9a-fA-F]+|[a-zA-Z]+);",
    regex=True,
    na=False
)

df_clean.loc[has_entities, "text_clean"] = (
    df_clean.loc[has_entities, "text_clean"]
    .astype("object")
    .map(html.unescape)
    .values
)

# Mark problematic texts as missing
df_clean.loc[problematic_text, "text_clean"] = pd.NA

# Classify text quality
df_clean["text_quality_status"] = "valid"
df_clean.loc[empty_text, "text_quality_status"] = "empty"
df_clean.loc[malformed_text, "text_quality_status"] = "malformed"

display(df_clean["text_quality_status"].value_counts())

print("Usable texts:", df_clean["text_clean"].notna().sum())
print("Unusable texts:", df_clean["text_clean"].isna().sum())


In [ ]:

assert df_clean.loc[problematic_text, "text_clean"].isna().all()
assert df_clean.loc[~problematic_text, "text_clean"].notna().all()

assert (
    df_clean["text_clean"].isna().sum()
    == problematic_text.sum()
)

print("Text quality validation passed.")


## 6. Timestamp Quality and Temporal Consistency

We examine publication and scraping timestamps to identify invalid dates, limited time precision, and chronological inconsistencies.

Ambiguous timestamps are flagged rather than artificially corrected.

In [ ]:

# Check timestamp ranges and missing values
display(pd.DataFrame({
    "Missing": df_clean[["timestamp", "time_scraped"]].isna().sum(),
    "Earliest": df_clean[["timestamp", "time_scraped"]].min(),
    "Latest": df_clean[["timestamp", "time_scraped"]].max()
}))

# Identify timestamps recorded at midnight
midnight_publication = (
    df_clean["timestamp"]
    .eq(df_clean["timestamp"].dt.normalize())
)

midnight_scraping = (
    df_clean["time_scraped"]
    .eq(df_clean["time_scraped"].dt.normalize())
)

print("Publication timestamps at midnight:", midnight_publication.sum())
print("Scraping timestamps at midnight:", midnight_scraping.sum())


In [ ]:

# Posts apparently published after scraping
inconsistent = df_clean["timestamp"] > df_clean["time_scraped"]

# Check whether the inconsistency occurs on the same day
same_day = (
    df_clean["timestamp"].dt.normalize()
    == df_clean["time_scraped"].dt.normalize()
)

# Separate ambiguous timestamps from actual inconsistencies
midnight_ambiguity = inconsistent & same_day & midnight_scraping
chronological_error = inconsistent & ~midnight_ambiguity

print("Publication after scraping:", inconsistent.sum())
print("Same-day midnight ambiguity:", midnight_ambiguity.sum())
print("Other chronological inconsistencies:", chronological_error.sum())

display(
    df_clean.loc[
        inconsistent,
        ["id", "timestamp", "time_scraped"]
    ].head(10)
)


In [ ]:

df_clean["temporal_quality_status"] = "valid"

df_clean.loc[
    midnight_ambiguity,
    "temporal_quality_status"
] = "midnight_ambiguity"

df_clean.loc[
    chronological_error,
    "temporal_quality_status"
] = "chronological_inconsistency"

df_clean.loc[
    df_clean["timestamp"].isna() | df_clean["time_scraped"].isna(),
    "temporal_quality_status"
] = "missing"

display(df_clean["temporal_quality_status"].value_counts())


In [ ]:

# Identify date-only values in the original dataset
date_only = df_raw["timestamp"].str.strip().str.fullmatch(
    r"\d{4}-\d{2}-\d{2}",
    na=False
)

print("Original date-only timestamps:", date_only.sum())
print("Original midnight timestamps:", (
    df_raw["timestamp"].str.contains(
        r"[ T]00:00:00(?:\.0+)?$",
        regex=True,
        na=False
    )
).sum())

print("Midnight timestamps after cleaning:", midnight_publication.sum())


In [ ]:

# Identify records with date-only publication timestamps
date_only_ids = df_raw.loc[date_only, "id"]

df_clean["timestamp_date_only"] = (
    df_clean["id"].astype("string").isin(date_only_ids)
)

print("Date-only timestamps preserved:",
      df_clean["timestamp_date_only"].sum())


## 7. Engagement Metrics and Outlier Detection

We analyze likes, retruths, and replies to identify invalid values, unusual distributions, and potential outliers.

High engagement values are not automatically considered errors, as viral posts can legitimately receive unusually large numbers of interactions.

In [ ]:

engagement_cols = ["like_count", "retruth_count", "reply_count"]

# Descriptive statistics
display(
    df_clean[engagement_cols].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ).round(2)
)

# Check invalid negative values
negative_counts = (df_clean[engagement_cols] < 0).sum()

print("Negative values:")
display(negative_counts)


In [ ]:

# Count posts with zero engagement
zero_counts = (df_clean[engagement_cols] == 0).sum()

zero_percentage = (
    zero_counts / len(df_clean) * 100
).round(2)

display(pd.DataFrame({
    "Zero values": zero_counts,
    "Percentage": zero_percentage
}))

# Posts with no recorded engagement
no_engagement = (df_clean[engagement_cols] == 0).all(axis=1)

print("Posts with zero engagement:", no_engagement.sum())
print("Percentage:", round(no_engagement.mean() * 100, 2))


### 7.1 Outlier Detection

We use the Interquartile Range (IQR) method to identify unusually high engagement values.

These outliers are analyzed but not removed, since they may represent genuine viral activity.

In [ ]:

outlier_summary = []

for col in engagement_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)

    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    outliers = (df_clean[col] > upper_bound).sum()

    outlier_summary.append({
        "Metric": col,
        "Upper bound": upper_bound,
        "Outliers": outliers,
        "Percentage": round(outliers / len(df_clean) * 100, 2)
    })

display(pd.DataFrame(outlier_summary))


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, engagement_cols):
    values = np.log1p(df_clean[col].dropna().astype(float))

    ax.hist(values, bins=50)
    ax.set_title(col)
    ax.set_xlabel("log(1 + count)")
    ax.set_ylabel("Posts")

plt.tight_layout()
plt.show()


In [ ]:

# Total recorded engagement
df_clean["total_engagement"] = df_clean[engagement_cols].sum(axis=1)

# Show the most engaged posts
display(
    df_clean.nlargest(10, "total_engagement")[
        ["id", "author", "timestamp",
         "like_count", "retruth_count",
         "reply_count", "total_engagement"]
    ]
)


In [ ]:

# Inspect posts sharing the highest engagement values
top_engagement = df_clean["total_engagement"].max()

top_posts = df_clean[
    df_clean["total_engagement"] == top_engagement
]

display(
    top_posts[
        ["id", "author", "is_retruth",
         "truth_retruthed", "like_count",
         "retruth_count", "reply_count"]
    ]
)

print("Posts sharing maximum engagement:", len(top_posts))
print("Unique original posts:", top_posts["truth_retruthed"].nunique())


## 8. Identifier Consistency

We investigate duplicated post IDs containing conflicting information.

Conflicting records are validated using author information from `users.xml`. Only records with verified author-username relationships are retained.

We also examine retruth references and their consistency with the main dataset.

In [ ]:

# Find records sharing the same ID
conflicts = df_clean[
    df_clean["id"].duplicated(keep=False)
].copy()

# Extract usernames from URLs
conflicts["username"] = conflicts["url"].str.extract(
    r"(?i)truthsocial\.com/@([^/]+)/posts/",
    expand=False
).str.lower()

# Validate authors using users.xml
conflicts["expected_author"] = conflicts["username"].map(users_map)

conflicts["author_match"] = (
    conflicts["author"] == conflicts["expected_author"]
).fillna(False)

display(
    conflicts[
        ["id", "author", "username",
         "expected_author", "author_match"]
    ].sort_values("id")
)

print("Conflicting IDs:", conflicts["id"].nunique())
print("Validated records:", conflicts["author_match"].sum())


In [ ]:

# Ensure each conflicting ID has one valid version
valid_versions = conflicts.groupby("id")["author_match"].sum()

assert (valid_versions == 1).all()

# Identify incorrect versions
invalid_indices = conflicts.index[
    ~conflicts["author_match"]
]

# Preserve excluded records for inspection
discarded_conflicts = df_clean.loc[invalid_indices].copy()

# Remove incorrect records
df_clean = df_clean.drop(index=invalid_indices).reset_index(drop=True)

print("Removed conflicting records:", len(discarded_conflicts))
print("Remaining records:", len(df_clean))
print("Duplicated IDs:", df_clean["id"].duplicated().sum())

assert df_clean["id"].is_unique


### 8.1 Retruth Reference Cleaning

The `truth_retruthed` column contains references to original posts, but also missing values and sentinel values such as `0` and `-1`.

We normalize the references and check whether the referenced posts exist in the dataset.

Missing references are preserved as missing values rather than replaced with invented identifiers.

In [ ]:

# Convert retruth references to numeric values
references = pd.to_numeric(
    df_clean["truth_retruthed"],
    errors="coerce"
).astype("Int64")

# Examine invalid and sentinel values
print("Empty or non-numeric references:", references.isna().sum())
print("Zero references:", (references == 0).sum())
print("Minus one references:", (references == -1).sum())

# Keep only positive identifiers
df_clean["truth_retruthed_clean"] = references.where(references > 0)

print(
    "Missing references after cleaning:",
    df_clean["truth_retruthed_clean"].isna().sum()
)


In [ ]:

# Check whether referenced posts exist
is_retruth = df_clean["is_retruth"].fillna(False)
reference = df_clean["truth_retruthed_clean"]

target_exists = reference.isin(df_clean["id"])
self_reference = reference.eq(df_clean["id"]).fillna(False)

# Classify references
df_clean["retruthed_reference_status"] = "not_retruth"

df_clean.loc[
    is_retruth & reference.isna(),
    "retruthed_reference_status"
] = "missing_reference"

df_clean.loc[
    is_retruth & reference.notna() & ~target_exists,
    "retruthed_reference_status"
] = "target_absent"

df_clean.loc[
    is_retruth & target_exists,
    "retruthed_reference_status"
] = "resolved"

df_clean.loc[
    is_retruth & self_reference,
    "retruthed_reference_status"
] = "self_reference"

df_clean.loc[
    ~is_retruth & reference.notna(),
    "retruthed_reference_status"
] = "inconsistent_non_retruth"

display(df_clean["retruthed_reference_status"].value_counts())


### 8.2 External ID and URL Consistency

We examine the consistency between `external_id` and `url`.

Some records contain usernames in `external_id` and numeric post identifiers in `url`. We attempt to reconstruct these fields using validated author-username relationships from `users.xml`.

Original values are preserved, and unresolved anomalies are documented.

In [ ]:

# Check external ID format
external_id = df_clean["external_id"].astype("string").str.strip()
url = df_clean["url"].astype("string").str.strip()

numeric_external = external_id.str.fullmatch(r"\d+", na=False).fillna(False)
numeric_url = url.str.fullmatch(r"\d+", na=False).fillna(False)

# Extract post IDs from standard URLs
url_post_id = url.str.extract(
    r"(?i)^https?://(?:www\.)?truthsocial\.com/@[^/]+/posts/(\d+)/?$",
    expand=False
)

valid_url = url_post_id.notna()

# Compare identifiers where possible
mismatch = (
    numeric_external
    & valid_url
    & external_id.ne(url_post_id).fillna(False)
)

print("Invalid external IDs:", (~numeric_external).sum())
print("Numeric-only URLs:", numeric_url.sum())
print("Other invalid URLs:", (~valid_url & ~numeric_url).sum())
print("External ID / URL mismatches:", mismatch.sum())

display(
    df_clean.loc[
        numeric_url,
        ["id", "author", "external_id", "url"]
    ].head(10)
)


In [ ]:

# Build an unambiguous author-to-username mapping
username_counts = users.groupby("id_user")["username"].nunique()
valid_authors = username_counts[username_counts == 1].index

author_map = (
    users[users["id_user"].isin(valid_authors)]
    .groupby("id_user")["username"]
    .first()
)

# Recover usernames using author IDs
username = df_clean["author"].map(author_map)

# Check that usernames stored in external_id are consistent
username_match = external_id.str.lower().eq(username).fillna(False)

# Select recoverable numeric URLs
recoverable = numeric_url & username.notna() & (
    numeric_external | username_match
)

print("Numeric URLs:", numeric_url.sum())
print("Recoverable URLs:", recoverable.sum())
print("Unresolved numeric URLs:", (numeric_url & ~recoverable).sum())


In [ ]:

# Preserve original identifiers
df_clean["external_id_clean"] = external_id.copy()
df_clean["url_clean"] = url.copy()

# Recover numeric post IDs
df_clean.loc[recoverable, "external_id_clean"] = url[recoverable]

# Reconstruct standard URLs
df_clean.loc[recoverable, "url_clean"] = (
    "https://truthsocial.com/@"
    + username[recoverable]
    + "/posts/"
    + url[recoverable]
)

print("Reconstructed URLs:", recoverable.sum())

display(
    df_clean.loc[
        recoverable,
        ["id", "external_id_clean", "url_clean"]
    ].head(10)
)


In [ ]:

# Validate cleaned identifiers
clean_external = df_clean["external_id_clean"].str.fullmatch(
    r"\d+", na=False
).fillna(False)

clean_url_id = df_clean["url_clean"].str.extract(
    r"(?i)^https?://(?:www\.)?truthsocial\.com/@[^/]+/posts/(\d+)/?$",
    expand=False
)

valid_clean_url = clean_url_id.notna()

# Check consistency between external IDs and URLs
mismatch = (
    clean_external
    & valid_clean_url
    & df_clean["external_id_clean"].ne(clean_url_id).fillna(False)
)

print("Valid external IDs:", clean_external.sum())
print("Valid URLs:", valid_clean_url.sum())
print("ID / URL mismatches:", mismatch.sum())

# Inspect unresolved records
unresolved = ~clean_external | ~valid_clean_url

print("Unresolved records:", unresolved.sum())

display(
    df_clean.loc[
        unresolved,
        ["id", "author", "external_id_clean", "url_clean"]
    ]
)


In [ ]:

# Recover the remaining numeric URL using its existing username
mask = df_clean["id"].eq(894769)

post_id = df_clean.loc[mask, "url_clean"].iloc[0]
username = df_clean.loc[mask, "external_id_clean"].iloc[0]

df_clean.loc[mask, "external_id_clean"] = post_id
df_clean.loc[mask, "url_clean"] = (
    f"https://truthsocial.com/@{username}/posts/{post_id}"
)

# Mark unrecoverable identifiers as missing
mask = df_clean["id"].eq(430702)

df_clean.loc[mask, ["external_id_clean", "url_clean"]] = pd.NA

print("Remaining invalid external IDs:",
      (~df_clean["external_id_clean"].str.fullmatch(
          r"\d+", na=False
      )).sum())

print("Remaining missing URLs:",
      df_clean["url_clean"].isna().sum())


In [ ]:

# Check duplicated external identifiers
duplicated_external = df_clean[
    df_clean["external_id_clean"].notna()
    & df_clean["external_id_clean"].duplicated(keep=False)
]

print("Duplicated external IDs:",
      duplicated_external["external_id_clean"].nunique())

print("Records involved:", len(duplicated_external))

display(
    duplicated_external[
        ["id", "author", "external_id_clean", "url_clean"]
    ].head(10)
)


In [ ]:

# Compare duplicated external IDs with retruth status
duplicated_mask = (
    df_clean["external_id_clean"].notna()
    & df_clean["external_id_clean"].duplicated(keep=False)
)

display(
    pd.crosstab(
        duplicated_mask,
        df_clean["is_retruth"],
        rownames=["Duplicated external ID"],
        colnames=["Is retruth"]
    )
)

# Check whether the same external ID is associated with different URLs
url_counts = (
    df_clean.loc[duplicated_mask]
    .groupby("external_id_clean")["url_clean"]
    .nunique()
)

print("External IDs with multiple URLs:", (url_counts > 1).sum())


## 9. Feature Extraction from Text

We extract basic textual features to enrich the dataset and support subsequent analyses.

These features describe text length, word count, hashtags, mentions, and external links. Missing or unusable texts are handled separately.

In [ ]:

# Text length in characters
df_clean["text_length"] = (
    df_clean["text_clean"].str.len().fillna(0).astype("Int64")
)

# Number of words
df_clean["word_count"] = (
    df_clean["text_clean"]
    .str.count(r"\b\w+\b")
    .fillna(0)
    .astype("Int64")
)

display(
    df_clean[["text_length", "word_count"]].describe()
)


In [ ]:

# Count hashtags, mentions and URLs in the text
patterns = {
    "hashtag_count": r"(?<![\w/])#[A-Za-z0-9_]+",
    "mention_count": r"(?<![\w/])@[A-Za-z0-9_]+",
    "url_count": r"https?://[^\s<>\"']+"
}

for col, pattern in patterns.items():
    df_clean[col] = (
        df_clean["text_clean"]
        .str.count(pattern)
        .fillna(0)
        .astype("Int64")
    )

display(
    df_clean[list(patterns)].describe()
)


In [ ]:

text_features = [
    "text_length",
    "word_count",
    "hashtag_count",
    "mention_count",
    "url_count"
]

# Summary of extracted features
summary = pd.DataFrame({
    "Mean": df_clean[text_features].mean().round(2),
    "Median": df_clean[text_features].median(),
    "Max": df_clean[text_features].max(),
    "Posts with feature": (df_clean[text_features] > 0).sum()
})

display(summary)


In [ ]:

# Inspect posts containing hashtags or mentions
sample = df_clean[
    (df_clean["hashtag_count"] > 0)
    | (df_clean["mention_count"] > 0)
]

display(
    sample[
        ["text_clean", "text_length", "word_count",
         "hashtag_count", "mention_count", "url_count"]
    ].head(5)
)


In [ ]:

# Check that all feature counts are non-negative
assert (df_clean[text_features] >= 0).all().all()

# Check that missing texts have zero extracted features
missing_text = df_clean["text_clean"].isna()

assert (df_clean.loc[missing_text, text_features] == 0).all().all()

print("Text feature validation passed.")
print("Total records:", len(df_clean))
print("Extracted features:", len(text_features))


## 10. Final Data Quality Validation

We validate the cleaned dataset before export, checking primary key uniqueness, missing values, engagement metrics, and the consistency of cleaned identifiers.

Unresolved anomalies are preserved or explicitly documented rather than corrected without sufficient evidence.

In [ ]:

# Check dataset structure
print("Total records:", len(df_clean))
print("Total columns:", df_clean.shape[1])

# Primary key
print("Missing IDs:", df_clean["id"].isna().sum())
print("Duplicated IDs:", df_clean["id"].duplicated().sum())

# Author recovery
print("Missing authors:", df_clean["author"].isna().sum())

# Invalid engagement values
print("Negative engagement values:")
display((df_clean[engagement_cols] < 0).sum())

assert df_clean["id"].is_unique
assert df_clean["id"].notna().all()
assert df_clean["author"].notna().all()

print("Structural validation passed.")


In [ ]:

# Check external identifiers and URLs
valid_external = df_clean["external_id_clean"].str.fullmatch(
    r"\d+", na=False
).fillna(False)

valid_url = df_clean["url_clean"].str.fullmatch(
    r"https?://(?:www\.)?truthsocial\.com/@[^/]+/posts/\d+/?",
    na=False
).fillna(False)

# Check retruth references
original_ref = pd.to_numeric(
    df_clean["truth_retruthed"], errors="coerce"
)

clean_ref = df_clean["truth_retruthed_clean"]

positive_ref = original_ref > 0

assert original_ref[positive_ref].eq(clean_ref[positive_ref]).all()
assert clean_ref.dropna().gt(0).all()

print("Invalid external IDs:", (~valid_external).sum())
print("Invalid or missing URLs:", (~valid_url).sum())
print("Missing retruth references:", clean_ref.isna().sum())

print("Identifier validation passed.")


In [ ]:

# Summary of unresolved data quality issues
quality_summary = {
    "Missing authors": df_clean["author"].isna().sum(),
    "Unusable texts": df_clean["text_clean"].isna().sum(),
    "Missing external IDs": df_clean["external_id_clean"].isna().sum(),
    "Missing URLs": df_clean["url_clean"].isna().sum(),
    "Missing retruth references": df_clean["truth_retruthed_clean"].isna().sum(),
    "Absent retruth targets": (
        df_clean["retruthed_reference_status"] == "target_absent"
    ).sum(),
    "Self-references": (
        df_clean["retruthed_reference_status"] == "self_reference"
    ).sum(),
    "Temporal ambiguities": (
        df_clean["temporal_quality_status"] == "midnight_ambiguity"
    ).sum()
}

display(
    pd.DataFrame(
        quality_summary.items(),
        columns=["Data quality issue", "Records"]
    )
)


### 10.1 Final Dataset Preparation

We prepare the cleaned dataset using the original column structure.

Cleaned text, external identifiers, URLs, and retruth references replace their original counterparts. Additional features and quality indicators are stored separately for future analysis.

In [ ]:

# Preserve the original column structure
original_columns = df_raw.columns.tolist()

df_final = df_clean[original_columns].copy()

# Apply cleaned values
df_final["text"] = df_clean["text_clean"]
df_final["external_id"] = df_clean["external_id_clean"]
df_final["url"] = df_clean["url_clean"]
df_final["truth_retruthed"] = df_clean["truth_retruthed_clean"]

print("Final dataset shape:", df_final.shape)

display(df_final.isna().sum())

assert df_final.columns.tolist() == original_columns
assert df_final["id"].is_unique
assert len(df_final) == 1038821


In [ ]:

# Store derived features and quality indicators
feature_columns = [
    "id",
    "has_media",
    "text_quality_status",
    "temporal_quality_status",
    "retruthed_reference_status",
    "total_engagement",
    "text_length",
    "word_count",
    "hashtag_count",
    "mention_count",
    "url_count"
]

df_features = df_clean[feature_columns].copy()

assert df_features["id"].is_unique

print("Feature dataset shape:", df_features.shape)
display(df_features.head())


### 10.2 Data Export

We export the cleaned Truths dataset and a separate file containing the derived features and quality indicators.

Both files are linked through the unique post identifier `id`.

In [ ]:

# Create output directory
OUTPUT_DIR = DATA_DIR.parent / "Cleaned_Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Export cleaned Truths
df_final.to_csv(
    OUTPUT_DIR / "truths_clean.csv",
    index=False
)

# Export derived features
df_features.to_csv(
    OUTPUT_DIR / "truths_features.csv",
    index=False
)

print("Files exported successfully:")
print("truths_clean.csv")
print("truths_features.csv")
